In [15]:
import time
import json
from api_client import TradingDeskAPI
from database import save_snapshots

counter = 0

def should_collect(market):

    prices = json.loads(market["outcomePrices"])
    yes_price = float(prices[0])

    if yes_price < 0.05 or yes_price > 0.95:
        return False

    return True

BTC_KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt",
    "bitcoin price",
    "bitcoin hits",
    "bitcoin above",
    "bitcoin below",
]

def is_btc_market(market):
    text = (
        market["question"] + " " +
        market.get("description", "")
    ).lower()

    return any(k in text for k in BTC_KEYWORDS)

api = TradingDeskAPI()

# {'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f'

def collect(counter):

    markets = api.get_markets()
    print(len(markets))

    snapshots = []

    for market in markets:

        if not is_btc_market(market):
            # print('not btc market')
            continue

        if not market["acceptingOrders"]:
            continue

        if market.get("ended"):
            continue

        if not market.get("enableOrderBook"):
            continue

        if not market.get("clobTokenIds"):
            print('no token ids')
            continue

        token_ids = json.loads(market["clobTokenIds"])
        print('token_ids', token_ids)
        yes_token, no_token = token_ids

        yes_book = api.get_orderbook(yes_token)
        no_book = api.get_orderbook(no_token)

        if not yes_book.get("bids") or not yes_book.get("asks") or not no_book.get("bids") or not no_book.get("asks"):
            print("empty orderbooks")
            continue

        print(market)
        print(market["outcomePrices"])

        best_bid_yes = float(yes_book["bids"][-1]["price"])
        best_ask_yes = float(yes_book["asks"][-1]["price"])


        print("YES ASKS")
        for x in yes_book["asks"]:
            print(x)

        print("YES BIDS")
        for x in yes_book["bids"]:
            print(x)

        print("NO ASKS")
        for x in no_book["asks"]:
            print(x)

        print("NO BIDS")
        for x in no_book["bids"]:
            print(x)
        

        best_bid_no = float(no_book["bids"][-1]["price"])
        best_ask_no = float(no_book["asks"][-1]["price"])

        spread_yes = best_ask_yes - best_bid_yes
        spread_no = best_ask_no - best_bid_no
        # if spread_yes > 0.03 or spread_no > 0.03:
        #     print('wide spread')
        #     continue

        snapshot = {
            "condition_id": market["conditionId"],
            "question": market["question"],
            "description": market["description"],
            "volume": market["volume"],
            "volumeNum": market["volumeNum"],
            "liquidityNum": market["liquidityNum"],
            "orderPriceMinTickSize": market["orderPriceMinTickSize"],
            "orderMinSize": market["orderMinSize"],
            "best_bid_yes": best_bid_yes,
            "best_ask_yes": best_ask_yes,
            "spread_yes": spread_yes,
            "price_yes": float(json.loads(market["outcomePrices"])[0]),
            "orderbook_yes": json.dumps(yes_book),
            "best_bid_no": best_bid_no,
            "best_ask_no": best_ask_no,
            "spread_no": spread_no,
            "price_no": float(json.loads(market["outcomePrices"])[1]),
            "orderbook_no": json.dumps(no_book),
            "taker_fee_rate": market["feeSchedule"]["rate"] if market["feesEnabled"] == True else 0,
            "events": json.dumps(market["events"]),
            "timestamp": time.time(),
            "token_yes": yes_token,
            "token_no": no_token,
        }

        # snapshot = { get snapshot using exchange data
        #     "question": market["question"],
        #     "model_probability": model_prob,
        #     "yes_bid": yes_bid,
        #     "yes_ask": yes_ask,
        #     "buy_yes_ev": result["buy_yes_ev"],
        #     "sell_yes_ev": result["sell_yes_ev"],
        #     "buy_no_ev": result["buy_no_ev"],
        #     "sell_no_ev": result["sell_no_ev"],
        # }

        snapshots.append(snapshot)

        print("question:", snapshot["question"])
        print("price_yes:", snapshot["price_yes"],)
        print("best_ask_yes:", snapshot["best_ask_yes"],)
        print("best_bid_yes:", snapshot["best_bid_yes"])
        print("price_no:", snapshot["price_no"])
        print("best_ask_no:", snapshot["best_ask_no"])
        print("best_bid_no:", snapshot["best_bid_no"])
        print("taker_fee_rate:", snapshot["taker_fee_rate"])
        break

    save_snapshots(snapshots, counter)

    return snapshots

try:
    snapshots = collect(counter)

except Exception as e:
    print("ERROR:", e)

# sorted(markets, key=lambda x: x["buy_no_ev"], reverse=True)


# while True:

#     try:
#         collect(counter)

#     except Exception as e:
#         print("ERROR:", e)
#     counter += 1

#     time.sleep(300)

100
token_ids ['56078938060096976448086754249497300447360333783952000147427828224794011030104', '11291662904897713174667903388388696640643610556195928998276904135282270136756']
{'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 'slug': 'will-bitcoin-reach-100000-by-december-31-2026-571-361-361', 'resolutionSource': '', 'endDate': '2027-01-01T05:00:00Z', 'liquidity': '94034.2151', 'startDate': '2025-11-24T19:07:17.691Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for Bitcoin (BTC/USDT) between November 24, 2025, 14:00 and December 31, 2026, 23:59 in the ET timezone has a final "High" price equal to or greater than the price specified in the title. Otherwise, this 

In [ ]:
{'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 'slug': 'will-bitcoin-reach-100000-by-december-31-2026-571-361-361', 
 'resolutionSource': '', 'endDate': '2027-01-01T05:00:00Z', 'liquidity': '94083.1351', 'startDate': '2025-11-24T19:07:17.691Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for Bitcoin (BTC/USDT) between November 24, 2025, 14:00 and December 31, 2026, 23:59 in the ET timezone has a final "High" price equal to or greater than the price specified in the title. Otherwise, this market will resolve to "No."\n\nThe resolution source for this market is Binance, specifically the BTC/USDT "High" prices available at https://www.binance.com/en/trade/BTC_USDT, with the chart settings on "1m" for one-minute candles selected on the top bar.\n\nPlease note that the outcome of this market depends solely on the price data from the Binance BTC/USDT trading pair. Prices from other exchanges, different trading pairs, or spot markets will not be considered for the resolution of this market.', 
'outcomes': '["Yes", "No"]', 'outcomePrices': '["0.095", "0.905"]', 'volume': '2370982.5583320004', 'active': True, 'closed': False, 'marketMakerAddress': '', 'createdAt': '2025-11-24T18:55:12.725029Z', 'updatedAt': '2026-08-02T10:54:52.526927Z', 
'new': False, 'featured': False, 'submitted_by': '0x91430CaD2d3975766499717fA0D66A78D814E5c5', 'archived': False, 'resolvedBy': '0x65070BE91477460D8A7AeEb94ef92fe056C2f2A7', 'restricted': True, 'groupItemTitle': '↑ 100,000', 'groupItemThreshold': '13', 
'questionID': '0x3c9be67d4b90291760ac3bffc1f9470a1966e5c1f3e99131333170e3469bd023', 'enableOrderBook': True, 'orderPriceMinTickSize': 0.01, 'orderMinSize': 5, 'volumeNum': 2370982.5583320004, 'liquidityNum': 94083.1351, 'endDateIso': '2027-01-01', 
'startDateIso': '2025-11-24', 'hasReviewedDates': True, 'volume24hr': 1365.610655, 'volume1wk': 66640.791618, 'volume1mo': 228733.97684700004, 'volume1yr': 2370982.5583320004, 
'clobTokenIds': '["56078938060096976448086754249497300447360333783952000147427828224794011030104", "11291662904897713174667903388388696640643610556195928998276904135282270136756"]', 
'comboStatus': 'disabled', 'umaBond': '500', 'umaReward': '5', 'volume24hrClob': 1365.610655, 'volume1wkClob': 66640.791618, 'volume1moClob': 228733.97684700004, 'volume1yrClob': 2370982.5583320004, 'volumeClob': 2370982.5583320004, 
'liquidityClob': 94083.1351, 'makerBaseFee': 1000, 'takerBaseFee': 1000, 'customLiveness': 0, 'acceptingOrders': True, 'negRisk': False, 'negRiskRequestID': '', 
'events': [{'id': '89502', 'ticker': 'what-price-will-bitcoin-hit-before-2027', 'slug': 'what-price-will-bitcoin-hit-before-2027', 
            'title': 'What price will Bitcoin hit in 2026?', 'description': 'What price will Bitcoin hit before 2027?  ', 
            'resolutionSource': '', 'startDate': '2025-11-24T19:07:12.848Z', 'creationDate': '2025-11-24T19:13:13.705687Z', 'endDate': '2027-01-01T05:00:00Z', 
            'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
            'active': True, 'closed': False, 'archived': False, 'new': False, 'featured': False, 'restricted': True, 'liquidity': 2695251.24076, 'volume': 50935482.262915, 
            'openInterest': 9503925.568983998, 'createdAt': '2025-11-24T18:55:05.597959Z', 'updatedAt': '2026-08-02T10:55:09.654318Z', 'competitive': 0.9999750006249843, 
            'volume24hr': 130499.32620200001, 'volume1wk': 2062901.9714630004, 'volume1mo': 7099616.726362999, 'volume1yr': 49102712.92022599, 'enableOrderBook': True, 
            'liquidityClob': 2695251.24076, 'negRisk': False, 'commentCount': 0, 'series': [{'id': '10016', 'ticker': 'bitcoin-hit-price-monthly', 'slug': 'bitcoin-hit-price-monthly', 
                                                                                        'title': 'Bitcoin Hit Price Monthly', 'seriesType': 'single', 'recurrence': 'monthly', 
                                                                                        'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'active': True, 'closed': False, 'archived': False, 'featured': False, 'restricted': True, 
                                                                                        'createdAt': '2025-01-31T22:03:50.00441Z', 'updatedAt': '2026-08-02T10:55:28.185077Z', 
                                                                                        'volume24hr': 601000.637578, 'volume': 51492794.133888, 'liquidity': 3438609.2653, 'commentCount': 6318, 
                                                                                        'requiresTranslation': False}], 
            'cyom': False, 'showAllOutcomes': True, 'showMarketImages': False, 'enableNegRisk': False, 'automaticallyActive': True, 'seriesSlug': 'bitcoin-hit-price-monthly', 
            'gmpChartMode': 'default', 'negRiskAugmented': False, 'estimateValue': True, 'cantEstimate': True, 'cumulativeMarkets': False, 'pendingDeployment': False, 'deploying': False, 
            'requiresTranslation': False, 'eventMetadata': {'context_requires_regen': True}, 'version': 'v1'}], 

'ready': False, 'funded': False, 'acceptingOrdersTimestamp': '2025-11-24T19:06:55Z', 
'cyom': False, 'competitive': 0.8590880780051975, 'pagerDutyNotificationEnabled': False, 'approved': True, 'clobRewards': [{'id': '418394', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 
                                                                                                                        'assetAddress': '0xc011a7e12a19f7b1f670d46f03b03f3342e82dfb', 'rewardsAmount': 0, 'rewardsDailyRate': 0.001, 
                                                                                                                        'startDate': '2026-06-03', 'endDate': '2500-12-31'}], 
'rewardsMinSize': 0, 'rewardsMaxSpread': 0, 'spread': 0.01, 'oneMonthPriceChange': -0.01, 'lastTradePrice': 0.09, 'bestBid': 0.09, 'bestAsk': 0.1, 'automaticallyActive': True, 
'clearBookOnStart': True, 'seriesColor': '', 'showGmpSeries': False, 'showGmpOutcome': False, 'manualActivation': False, 'negRiskOther': False, 'umaResolutionStatuses': '[]', 
'pendingDeployment': False, 'deploying': False, 'deployingTimestamp': '2025-11-24T19:06:23.727362Z', 'rfqEnabled': False, 'holdingRewardsEnabled': True, 'feesEnabled': True, 
'requiresTranslation': False, 'feeType': 'crypto_fees_v2', 'feeSchedule': {'exponent': 1, 'rate': 0.07, 'takerOnly': True, 'rebateRate': 0.2}, 'version': 'v1'}
["0.095", "0.905"]

In [ ]:
def calculate_ev(
        model_prob,
        best_ask_yes,
        best_bid_yes,
        best_ask_no,
        best_bid_no,
        fee_rate=0.07,
    ):
    """
    Expected PnL per contract for Polymarket.

    model_prob : P(YES)
    fee_rate   : taker fee rate (e.g. 0.07 for crypto markets)

    Assumes:
      - you are a taker
      - fee = fee_rate * price * (1 - price)
      - settlement has no fee
    """

    p_yes = model_prob
    p_no = 1 - p_yes

    def fee(price):
        #fee = C × feeRate × p × (1 - p)
        #Where C = number of shares traded and p = price of the shares.
        return fee_rate * price * (1 - price)

    # -------------------------
    # BUY YES
    # -------------------------
    buy_yes_cost = best_ask_yes + fee(best_ask_yes)

    profit_if_yes = 1 - buy_yes_cost
    cost_if_no = buy_yes_cost

    # EV: profit if yes - cost if no
    buy_yes_ev = p_yes * profit_if_yes - p_no * cost_if_no

    # -------------------------
    # SELL YES (short YES)
    # -------------------------
    sell_yes_credit = best_bid_yes - fee(best_bid_yes)

    profit_if_no = sell_yes_credit
    cost_if_yes = 1 - sell_yes_credit # need to pay the remaining of the $1 out of the credit you got, if yes happened

    # EV: profit if yes - cost if no
    sell_yes_ev = p_no * profit_if_no - p_yes * cost_if_yes

    # -------------------------
    # BUY NO
    # -------------------------
    buy_no_cost = best_ask_no + fee(best_ask_no)

    profit_if_no = 1 - buy_no_cost
    cost_if_yes = buy_no_cost

    buy_no_ev = p_no * profit_if_no - p_yes * cost_if_yes
   

    # -------------------------
    # SELL NO (short NO)
    # -------------------------
    sell_no_credit = best_bid_no - fee(best_bid_no)

    profit_if_yes = sell_no_credit
    cost_if_no = 1 - sell_no_credit

    sell_no_ev = p_yes * profit_if_yes - p_no * cost_if_no

    print("buy_yes_ev:", buy_yes_ev)
    print("sell_yes_ev:", sell_yes_ev)
    print("buy_no_ev:", buy_no_ev)
    print("sell_no_ev:", sell_no_ev)

    return {
        "buy_yes_ev": buy_yes_ev,
        "sell_yes_ev": sell_yes_ev,
        "buy_no_ev": buy_no_ev,
        "sell_no_ev": sell_no_ev,
        "buy_yes_fee": fee(best_ask_yes),
        "sell_yes_fee": fee(best_bid_yes),
        "buy_no_fee": fee(best_ask_no),
        "sell_no_fee": fee(best_bid_no),
    }


result = calculate_ev(
    model_prob=0.059,
    best_ask_yes=0.1,
    best_bid_yes=0.09,
    best_ask_no=0.91,
    best_bid_no=0.9,
    fee_rate=0.07
)

# for k,v in result.items():
#     print(k, v)

buy_yes_ev: -0.04730000000000002
sell_yes_ev: 0.025266999999999998
buy_no_ev: 0.025266999999999984
sell_no_ev: -0.04729999999999996


In [ ]:
ENTRY_THRESHOLD = 0.03
EXIT_THRESHOLD = 0.01

edge = 0.031

if edge > ENTRY_THRESHOLD:
    pass
    #sell_yes()

if edge < EXIT_THRESHOLD:
    pass
    #close_position()

In [ ]:
#                  Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4